In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

In [2]:
def write_to_submit_file(predicted_labels, out_file, target="target", index_label="session_id"):
    predicted_df = pd.DataFrame(predicted_labels, 
                                index=np.arange(1, predicted_labels.shape[0]+1),
                                columns=[target])
    predicted_df.to_csv(out_file, index_label=index_label)

In [4]:
train_df = pd.read_csv("./train_sessions.csv/train_sessions.csv", index_col="session_id")
test_df = pd.read_csv("./test_sessions.csv/test_sessions.csv", index_col="session_id")

In [7]:
train_df.columns

Index(['site1', 'time1', 'site2', 'time2', 'site3', 'time3', 'site4', 'time4',
       'site5', 'time5', 'site6', 'time6', 'site7', 'time7', 'site8', 'time8',
       'site9', 'time9', 'site10', 'time10', 'target'],
      dtype='object')

In [15]:
# convert time1, ...., time10 columns to datetime type

# times = ["time%s" % i for i in range (1,11)]
# times = ["time{}".format(i) for i in range(1, 11)]
times = [f"time{i}" for i in range(1, 11)]
train_df[times] = train_df[times].apply(pd.to_datetime)
test_df[times] = test_df[times].apply(pd.to_datetime)

# sort the data by time
train_df = train_df.sort_values(by="time1")

train_df.head()

,site1,time1,site2,time2,site3,time3,site4,time4,site5,time5,...,time6,site7,time7,site8,time8,site9,time9,site10,time10,target
session_id,,,,,,,,,,,,,,,,,,,,,
21669,56,2013-01-12 08:05:57,55.0,2013-01-12 08:05:57,NaN,NaT,NaN,NaT,NaN,NaT,...,NaT,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0
54843,56,2013-01-12 08:37:23,55.0,2013-01-12 08:37:23,56.0,2013-01-12 09:07:07,55.0,2013-01-12 09:07:09,NaN,NaT,...,NaT,NaN,NaT,NaN,NaT,NaN,NaT,NaN,NaT,0
77292,946,2013-01-12 08:50:13,946.0,2013-01-12 08:50:14,951.0,2013-01-12 08:50:15,946.0,2013-01-12 08:50:15,946.0,2013-01-12 08:50:16,...,2013-01-12 08:50:16,948.0,2013-01-12 08:50:16,784.0,2013-01-12 08:50:16,949.0,2013-01-12 08:50:17,946.0,2013-01-12 08:50:17,0
114021,945,2013-01-12 08:50:17,948.0,2013-01-12 08:50:17,949.0,2013-01-12 08:50:18,948.0,2013-01-12 08:50:18,945.0,2013-01-12 08:50:18,...,2013-01-12 08:50:18,947.0,2013-01-12 08:50:19,945.0,2013-01-12 08:50:19,946.0,2013-01-12 08:50:19,946.0,2013-01-12 08:50:20,0
146670,947,2013-01-12 08:50:20,950.0,2013-01-12 08:50:20,948.0,2013-01-12 08:50:20,947.0,2013-01-12 08:50:21,950.0,2013-01-12 08:50:21,...,2013-01-12 08:50:21,946.0,2013-01-12 08:50:21,951.0,2013-01-12 08:50:22,946.0,2013-01-12 08:50:22,947.0,2013-01-12 08:50:22,0


In [11]:
# Transform the data into format that can be fed to CountVectorizer
sites = [f"site{i}" for i in range (1,11)] 

train_df[sites].fillna(0).astype('int').to_csv("train_session_text.txt", sep=" ", index=None, header=None)
test_df[sites].fillna(0).astype('int').to_csv("test_session_text.txt", sep=" ", index=None, header=None)

In [16]:
with open("train_session_text.txt") as f:
    for _ in range(5):
        print(f.readline().rstrip())

56 55 0 0 0 0 0 0 0 0
56 55 56 55 0 0 0 0 0 0
946 946 951 946 946 945 948 784 949 946
945 948 949 948 945 946 947 945 946 946
947 950 948 947 950 952 946 951 946 947


In [20]:
cv = CountVectorizer()
# it will be called sparse metrix
# call todense to call the full metrix
X_sparse = cv.fit_transform(["this movie is awful",
                  "enjoyed this movie"])
X_sparse.todense()

matrix([[1, 0, 1, 1, 1],
        [0, 1, 0, 1, 1]])

In [ ]:
cv.vocabulary_
# this is the last one on the matrix, awful is the first one 

{'this': 4, 'movie': 3, 'is': 2, 'awful': 0, 'enjoyed': 1}

In [ ]:
X_sparse.indices
X_sparse.data
X_sparse.nonzero() # -> shows the row and column indices of non zero elements

array([4, 3, 2, 0, 4, 3, 1], dtype=int32)